In [1]:
from gevent import monkey
monkey.patch_all()
from gevent.queue import Queue
import gevent

# = = = = = = = = = = = = = = = = = =

import pandas as pd

import sys
sys.path.append('C:/Users/YangTeng/Desktop/venv/00.module')
import UA
import Proxy

# = = = = = = = = = = = = = = = = = =

from selenium.webdriver import ChromeOptions
option = ChromeOptions()
option.add_extension(Proxy.get_Proxy_Selenium())
option.add_experimental_option('useAutomationExtension', False)
option.add_argument('blink-settings=imagesEnabled=false')
option.add_argument('--disable-blink-features=AutomationControlled')
option.add_argument('--disable-dev-shm-usage')
option.add_argument('--disable-gpu')
option.add_experimental_option('excludeSwitches', ['enable-automation'])
# option.add_argument('--headless')
option.add_argument('--no-sandbox')
option.add_argument(UA.get_User_Agent_Selenium())
option.page_load_strategy = 'eager'

from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
import time
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By

C:\Users\YangTeng\Desktop\venv\lib\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [2]:
df_menu = pd.read_excel(r'C:\Users\YangTeng\Downloads\Summit_menu.xlsx', header=0)
list_url = df_menu['Url'].to_list()

print('总数量：' + str(len(list_url)))
print()

work = Queue()
for url in list_url:
    work.put_nowait(url)

# = = = = = = = = = = = = = = = = = =

df = pd.DataFrame(columns=['Url',
                           'Price',
                           'Src',
                           'status'])

总数量：745



C:\Users\YangTeng\Desktop\venv\lib\site-packages\gevent\hub.py:161: UserWarning: libuv only supports millisecond timer resolution; all times less will be set to 1 ms
  with loop.timer(seconds, ref=ref) as t:


In [3]:

def crawler():
    global df
    
    while not work.empty():
        url = work.get_nowait()
        
        status = 'error'
        for _ in range(3):
            try:
                browser = Chrome(service=Service(r'C:\Users\YangTeng\Desktop\venv\chromedriver.exe'), options=option)
                browser.maximize_window()
                
                # = = = = = = = = = = = = = = = = = =

                browser.get(url)
                
                time.sleep(5)
                
                # = = = = = = = = = = = = = = = = = =
                
                wait = WebDriverWait(browser, 60)
                wait.until(EC.presence_of_element_located((By.XPATH, '//p[@class="price"]')))
                price = browser.find_element(by=By.XPATH, value='//p[@class="price"]').text

                # = = = = = = = = = = = = = = = = = =
                
                wait.until(EC.presence_of_element_located((By.XPATH, '//img[@id="primaryImage"]')))
                src = browser.find_element(by=By.XPATH, value='//img[@id="primaryImage"]').get_attribute('src')
                
                # = = = = = = = = = = = = = = = = = =
                
                status = 'ok'
                df_temp = pd.DataFrame([{'Url': url,
                                         'Price': price,
                                         'Src': src,
                                         'status': 'ok'}])
                
                # = = = = = = = = = = = = = = = = = =

                wait.until(EC.presence_of_element_located((By.XPATH, '//div[@class="attribute-container flex-container flex-wrap"]')))
                list_name = [name.text for name in browser.find_elements(by=By.XPATH, value='//div[@class="attribute-container flex-container flex-wrap"]/div[@class="small-12 medium-9 large-6"]/strong')]
                list_value = [value.text for value in browser.find_elements(by=By.XPATH, value='//div[@class="attribute-container flex-container flex-wrap"]/div[@class="small-12 medium-15 large-18"]')]

                for name, value in zip(list_name, list_value):
                    df_temp[name[:-1].lower()] = value
                        
                browser.quit()

                break

            except:
                time.sleep(3)
                continue

        # = = = = = = = = = = = = = = = = = =

        if status == 'error':
            df_temp = pd.DataFrame([{'Url': url,
                                     'status': 'error'}])

        df = pd.concat([df, df_temp], ignore_index=True).fillna('')
        
        print(url +  '  <->  [' + status + '] - 剩余数量：' + str(work.qsize()))
        

In [ ]:
list_task = []
for _ in range(1):
    task = gevent.spawn(crawler)
    list_task.append(task)
gevent.joinall(list_task)

df.to_excel(r'C:\Users\YangTeng\Desktop\code\result--venv\7.Summit\Summit.xlsx', index=False)
print()
print('搞定')

https://www.summitracing.com//parts/aaf-all26240  <->  [ok] - 剩余数量：744
